In [20]:
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [21]:
load_dotenv()

gemini_api_key = os.getenv('GEMINI_API_KEY')

OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

if gemini_api_key:
    print(f"Gemini API Key exists")
else:
    print("Gemini API Key not set (and this is optional)")

Gemini API Key exists


In [16]:
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini = OpenAI(api_key=gemini_api_key, base_url=gemini_url)

In [53]:
ollama_model = "llama3.2"
gemini_model = "gemini-2.5-flash"

ollama_prompt = """
You are a poor farmer with two sons who eat a lot and do not want to work, teach them a lesson

Rules:
- Max 2 sentences per reply
- No explanations longer than one idea
- Stay in character
- Always react to the previous speaker
"""

gemini_prompt = """
You are a lazy son, you father is scolding your brother and you, defend yourself in a snarky and humorous manner.

Rules:
- Max 3 sentences
- Be concise and neutral
- Respond to the last message
- Do not repeat jokes
"""

ollama_intern_prompt = """
You are a lazy son, you father is scolding your brother and you, defend yourself in a snarky and humorous manner.

Rules:
- Max 2 sentences
- Always assert confidence
- Never admit you are wrong
- Always reference something vague you “read”
"""
ollama_message = ["hello"]
gemini_message = ["Hi"]
ollama_intern_message = ["Hey Guys."]

In [57]:
def call_llama():
    messages = [{"role": "system", "content": ollama_prompt}]
    for ollama_text, gemini_text, inter_text in zip(ollama_message, gemini_message, ollama_intern_message):
          messages.append({"role": "assistant", "content": ollama_text})
          messages.append({"role": "user", "content": gemini_text})
          messages.append({"role": "user", "content": inter_text})
    response = ollama.chat.completions.create(model=ollama_model, messages=messages)
    return response.choices[0].message.content

In [58]:
def call_llama_intern():
    messages = [{"role": "system", "content": ollama_intern_prompt}]

    for o, g, i in zip(ollama_message, gemini_message, ollama_intern_message):
        messages.append({"role": "user", "content": o})
        messages.append({"role": "user", "content": g})
        messages.append({"role": "assistant", "content": i})
    messages.append({"role": "user", "content": ollama_message[-1]})

    response = ollama.chat.completions.create(
        model=ollama_model,
        messages=messages
    )
    return response.choices[0].message.content


In [59]:
def call_gemini():
    messages = [{"role": "system", "content": gemini_prompt}]
    for ollama_text, gemini_text, inter_text in zip(ollama_message, gemini_message, ollama_intern_message):
        messages.append({"role": "assistant", "content": gemini_text})
        messages.append({"role": "user", "content": ollama_text})
        messages.append({"role": "user", "content": inter_text})
    messages.append({"role": "user", "content": ollama_message[-1]})
    messages.append({"role": "user", "content": ollama_intern_message[-1]})
    response = gemini.chat.completions.create(model=gemini_model, messages=messages)
    return response.choices[0].message.content

In [ ]:
ollama_message = ["Hi there"]
gemini_message = ["Hi"]
ollama_intern_message = ["Hey Guys."]

display(Markdown(f"### Ollama:\n{ollama_message[0]}\n"))
display(Markdown(f"### Ollama:\n{ollama_intern_message[0]}\n"))
display(Markdown(f"### Gemini:\n{gemini_message[0]}\n"))

for i in range(3):

    # Ollama (snarky)
    ollama_next = call_llama()
    display(Markdown(f"### Ollama:\n{ollama_next}\n"))
    ollama_message.append(ollama_next)

    # Gemini (calm)
    gemini_next = call_gemini()
    display(Markdown(f"### Gemini:\n{gemini_next}\n"))
    gemini_message.append(gemini_next)

    # Ollama Intern (annoying)
    intern_next = call_llama_intern()
    display(Markdown(f"### Intern:\n{intern_next}\n"))
    ollama_intern_message.append(intern_next)
